# Data Download and Initial Preprocessing

This notebook is the first step of the pipeline developed for the TFM.

Its objective is to download the historical time series of the IBEX 35 index, validate the integrity of the dataset, construct daily log returns and export a clean base dataset that will be used in the next stage of the project.

This notebook performs four main tasks:

1. Download the IBEX 35 historical series
2. Validate the structure and integrity of the downloaded data
3. Construct daily log returns
4. Export a clean dataset for further feature engineering

The final output of this notebook is a clean dataset saved as:

`data/raw/ibex_prices.csv`

This file is designed to be reusable in the next notebook and directly compatible with external tools such as Power BI.

## 1. Environment setup

We start by importing the libraries required for data manipulation, financial data download and project path management.

The notebook is designed to be fully reproducible and portable across machines by using relative paths only.

In [93]:
import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path

## 2. Project paths

To guarantee reproducibility, all file paths are defined using relative paths.

In [94]:
from pathlib import Path

# Robust project root detection
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)

Project root: /home/apalo/projects/TMF-Market-Regimes-Risk-Portfolio
Raw data directory: /home/apalo/projects/TMF-Market-Regimes-Risk-Portfolio/data/raw


## 3. Dataset parameters

We define the key parameters used in the download process.

The historical series of the IBEX 35 index will be downloaded from Yahoo Finance. The dataset will begin in January 2002 and will extend to the latest available market observation at the time of execution.

In [95]:
TICKER = "^IBEX"
START_DATE = "2002-01-01"
PRICE_COLUMN = "Adj Close"
OUTPUT_FILENAME = "ibex_prices.csv"

## 4. Data download

The IBEX 35 time series is downloaded using the `yfinance` interface to Yahoo Finance.

For the computation of returns we use the adjusted closing price. The adjusted price corrects the raw closing price for corporate actions such as dividends or stock splits, providing a consistent basis for return calculation over long time horizons.

In [96]:
ibex = yf.download(TICKER, start=START_DATE, progress=False)

In [97]:
print("Columns returned by Yahoo Finance:")
print(ibex.columns)

Columns returned by Yahoo Finance:
MultiIndex([( 'Close', '^IBEX'),
            (  'High', '^IBEX'),
            (   'Low', '^IBEX'),
            (  'Open', '^IBEX'),
            ('Volume', '^IBEX')],
           names=['Price', 'Ticker'])


## 5. Initial inspection

We inspect the raw structure of the downloaded dataset to verify that the download was successful and that the expected columns are available.

In [98]:
print("Downloaded columns:", list(ibex.columns))
ibex.head()

Downloaded columns: [('Close', '^IBEX'), ('High', '^IBEX'), ('Low', '^IBEX'), ('Open', '^IBEX'), ('Volume', '^IBEX')]


Price,Close,High,Low,Open,Volume
Ticker,^IBEX,^IBEX,^IBEX,^IBEX,^IBEX
Date,,,,,
2002-01-02,8377.090820,8446.590747,8247.691542,8397.590799,80800
2002-01-03,8554.691406,8575.790994,8408.791166,8408.791166,99400
2002-01-04,8463.090820,8608.491060,8416.291064,8554.691311,125000
2002-01-07,8177.291504,8506.191561,8156.591818,8463.091018,143200
2002-01-08,8186.591797,8262.392013,8105.091880,8177.291513,126600


## 6. Selection of the price series

For the empirical pipeline we retain only the adjusted closing price.

The selected series is renamed as `ibex_close`.

In [99]:
# Handle possible MultiIndex columns returned by yfinance
if isinstance(ibex.columns, pd.MultiIndex):
    first_level = ibex.columns.get_level_values(0)

    if "Adj Close" in first_level:
        ibex = ibex[[("Adj Close", TICKER)]].copy()
        ibex.columns = ["ibex_close"]
    elif "Close" in first_level:
        ibex = ibex[[("Close", TICKER)]].copy()
        ibex.columns = ["ibex_close"]
    else:
        raise ValueError("No suitable price column found in downloaded dataset.")
else:
    if "Adj Close" in ibex.columns:
        ibex = ibex[["Adj Close"]].rename(columns={"Adj Close": "ibex_close"})
    elif "Close" in ibex.columns:
        ibex = ibex[["Close"]].rename(columns={"Close": "ibex_close"})
    else:
        raise ValueError("No suitable price column found in downloaded dataset.")

## 7. Basic validation of the downloaded dataset

Before constructing returns, we validate the integrity of the series.

The checks include:

- start date
- end date
- number of observations
- missing values
- duplicate timestamps
- temporal ordering

In [100]:
print("Start date:", ibex.index.min())
print("End date:", ibex.index.max())
print("Number of observations:", len(ibex))
print("Missing values:\n", ibex.isna().sum())
print("Duplicate dates:", ibex.index.duplicated().sum())
print("Is index monotonic increasing?:", ibex.index.is_monotonic_increasing)

Start date: 2002-01-02 00:00:00
End date: 2026-03-13 00:00:00
Number of observations: 6165
Missing values:
 ibex_close    0
dtype: int64
Duplicate dates: 0
Is index monotonic increasing?: True


### Dataset coverage

The downloaded dataset spans from the earliest available observation to the most recent market data available at the time of execution.

In [101]:
print(f"Dataset covers {ibex.index.min().date()} to {ibex.index.max().date()}")

Dataset covers 2002-01-02 to 2026-03-13


## 8. Sorting and basic cleaning

Although financial downloads are usually ordered correctly, we sort the dataset by date to guarantee chronological consistency.

In [102]:
ibex = ibex.sort_index()

## 9. Construction of daily log returns

Daily log returns are computed from the adjusted closing price.

$$
r_t = \ln \left(\frac{P_t}{P_{t-1}}\right)
$$

where $P_t$ denotes the adjusted closing price of the IBEX 35 at time $t$.

In [103]:
display(ibex.head())
display(ibex.tail())

,ibex_close
Date,
2002-01-02,8377.090820
2002-01-03,8554.691406
2002-01-04,8463.090820
2002-01-07,8177.291504
2002-01-08,8186.591797


,ibex_close
Date,
2026-03-09,16928.199219
2026-03-10,17445.000000
2026-03-11,17351.900391
2026-03-12,17139.900391
2026-03-13,17140.699219


In [104]:
ibex["ret_1d"] = np.log(ibex["ibex_close"] / ibex["ibex_close"].shift(1))

## 10. Handling initial missing values

The first return observation is undefined because return calculation requires one lagged price observation.

We remove that initial missing value to obtain a clean dataset.

In [105]:
ibex = ibex.dropna()

In [106]:
print("Return statistics")
print(ibex["ret_1d"].describe())

Return statistics
count    6164.000000
mean        0.000116
std         0.013799
min        -0.151512
25%        -0.006422
50%         0.000698
75%         0.006957
max         0.134836
Name: ret_1d, dtype: float64


## 11. Final validation of the transformed dataset

We run a final validation step after return construction to ensure that the exported dataset is complete and internally consistent.

In [107]:
print("Final start date:", ibex.index.min())
print("Final end date:", ibex.index.max())
print("Final number of observations:", len(ibex))
print("Missing values after preprocessing:\n", ibex.isna().sum())
print("Duplicate dates after preprocessing:", ibex.index.duplicated().sum())

Final start date: 2002-01-03 00:00:00
Final end date: 2026-03-13 00:00:00
Final number of observations: 6164
Missing values after preprocessing:
 ibex_close    0
ret_1d        0
dtype: int64
Duplicate dates after preprocessing: 0


## 12. Descriptive preview

We inspect a short preview and basic descriptive statistics of the cleaned dataset before export.

In [108]:
display(ibex.head())
display(ibex.tail())
display(ibex.describe())

,ibex_close,ret_1d
Date,,
2002-01-03,8554.691406,0.020979
2002-01-04,8463.090820,-0.010765
2002-01-07,8177.291504,-0.034353
2002-01-08,8186.591797,0.001137
2002-01-09,8066.091797,-0.014829


,ibex_close,ret_1d
Date,,
2026-03-09,16928.199219,-0.008599
2026-03-10,17445.000000,0.030072
2026-03-11,17351.900391,-0.005351
2026-03-12,17139.900391,-0.012293
2026-03-13,17140.699219,0.000047


,ibex_close,ret_1d
count,6164.000000,6164.000000
mean,9864.892881,0.000116
std,2255.990441,0.013799
min,5364.494629,-0.151512
25%,8364.643799,-0.006422
50%,9433.649902,0.000698
75%,10854.692139,0.006957
max,18496.599609,0.134836


In [109]:
print("Skewness:", ibex["ret_1d"].skew())
print("Kurtosis:", ibex["ret_1d"].kurtosis())

Skewness: -0.3517246129611425
Kurtosis: 9.29636970447018


## 13. Preparing the dataset for external tools

To ensure compatibility with downstream tools such as Power BI, the date index is converted into an explicit column.

In [110]:
ibex = ibex.reset_index()
ibex.rename(columns={"Date": "date"}, inplace=True)

# Ensure date column is datetime
ibex["date"] = pd.to_datetime(ibex["date"])

In [111]:
# Ensure flat column names after reset_index
if isinstance(ibex.columns, pd.MultiIndex):
    ibex.columns = [col[0] if isinstance(col, tuple) else col for col in ibex.columns]

print("Final columns:", list(ibex.columns))

Final columns: ['date', 'ibex_close', 'ret_1d']


In [112]:
assert ibex["date"].is_monotonic_increasing, "Dates are not ordered"

## 14. Final schema check

We verify the final structure of the dataset before exporting it.

In [113]:
print("Final columns:", list(ibex.columns))
print("\nData types:")
print(ibex.dtypes)
ibex.head()

Final columns: ['date', 'ibex_close', 'ret_1d']

Data types:
date          datetime64[s]
ibex_close          float64
ret_1d              float64
dtype: object


,date,ibex_close,ret_1d
0,2002-01-03,8554.691406,0.020979
1,2002-01-04,8463.090820,-0.010765
2,2002-01-07,8177.291504,-0.034353
3,2002-01-08,8186.591797,0.001137
4,2002-01-09,8066.091797,-0.014829


## 15. Exporting the dataset

### Final dataset validation

The cleaned and reproducible dataset is exported as a CSV file.


In [114]:
print("Final dataset shape:", ibex.shape)
print("Date range:", ibex["date"].min(), "to", ibex["date"].max())

Final dataset shape: (6164, 3)
Date range: 2002-01-03 00:00:00 to 2026-03-13 00:00:00


The export uses `index=False` so that no artificial index column is added to the file. This keeps the dataset clean and directly compatible with:

- the next notebook in the pipeline
- Power BI
- spreadsheet tools
- future dashboard implementations

In [115]:
output_path = RAW_DATA_DIR / OUTPUT_FILENAME
ibex.to_csv(output_path, index=False)

print("Dataset exported successfully.")
print("Output path:", output_path)
print("Rows exported:", len(ibex))
print("Columns exported:", len(ibex.columns))

Dataset exported successfully.
Output path: /home/apalo/projects/TMF-Market-Regimes-Risk-Portfolio/data/raw/ibex_prices.csv
Rows exported: 6164
Columns exported: 3


## 16. Notebook summary

In [ ]:
print(f"Dataset exported to: {output_path}")
print(f"Final shape: {ibex.shape}")
print(f"Date coverage: {ibex['date'].min().date()} to {ibex['date'].max().date()}")
print(f"Columns: {list(ibex.columns)}")

Notebook 01 completed successfully.
Dataset exported to: /home/apalo/projects/TMF-Market-Regimes-Risk-Portfolio/data/raw/ibex_prices.csv
Final shape: (6164, 3)
Date coverage: 2002-01-03 to 2026-03-13
Columns: ['date', 'ibex_close', 'ret_1d']
